In [1]:
pip install cooper-optim


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import torch
import torch.nn as nn
import cooper
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:

# =========================================================
# Utilities
# =========================================================

def compute_d(arch):
    d = 0
    for i in range(len(arch) - 1):
        d += arch[i] * arch[i + 1]
    return d


def unpack_weights(w_flat, arch):
    matrices = []
    idx = 0
    for i in range(len(arch) - 1):
        n_in, n_out = arch[i], arch[i + 1]
        size = n_in * n_out
        W = w_flat[idx: idx + size].reshape(n_out, n_in)
        matrices.append(W)
        idx += size
    return matrices


def forward_pass(u, w_flat, biases, arch):
    weight_mats = unpack_weights(w_flat, arch)
    a = u
    for l, W in enumerate(weight_mats):
        z = W @ a + biases[l].unsqueeze(1)
        if l < len(weight_mats) - 1:
            a = torch.tanh(z)
        else:
            a = z
    return a


def binary_entropy(Q, eps=1e-8):
    Qc = Q.clamp(eps, 1 - eps) # 
    return -(Qc * Qc.log() + (1 - Qc) * (1 - Qc).log()).sum()


# =========================================================
# Model
# =========================================================

class SparseNNModel(nn.Module):
    def __init__(self, arch, k, eps=1e-6):
        super().__init__()
        self.arch = arch
        self.k = k
        self.eps = eps
        self.d = compute_d(arch)

        # Q initialized column-stochastic
        Q_init = torch.ones(self.d, k) / self.d
        self.Q = nn.Parameter(Q_init)

        # reduced coefficients
        self.x_coeff = nn.Parameter(torch.randn(k) * 0.1)

        # biases
        self.biases = nn.ParameterList()
        for i in range(len(arch) - 1):
            self.biases.append(nn.Parameter(torch.zeros(arch[i + 1])))

    def get_w(self):
        return self.Q @ self.x_coeff

    def forward(self, u):
        return forward_pass(u, self.get_w(), self.biases, self.arch)

    def clamp_Q(self):
        with torch.no_grad():
            self.Q.clamp_(self.eps, 1.0 - self.eps)


# =========================================================
# Cooper CMP
# =========================================================

class SparseNNCMP(cooper.ConstrainedMinimizationProblem):
    def __init__(
        self,
        model,
        c0,
        u_data,
        y_data,
        init_penalty_entropy=1.0,
        init_penalty_colsum=5.0,
        tol=1e-6,
    ):
        super().__init__()
        self.model = model
        self.c0 = c0
        self.u_data = u_data
        self.y_data = y_data

        # H(Q) - c0 <= 0
        self.entropy_eq = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=1),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.tensor([init_penalty_entropy], dtype=torch.float32)
            ),
        )

        # Q^T 1_d - 1_k - eps <= 0
        self.colsum_eq_upper = cooper.Constraint(
            constraint_type=cooper.ConstraintType.INEQUALITY,
            formulation_type=cooper.formulations.AugmentedLagrangian,
            multiplier=cooper.multipliers.DenseMultiplier(num_constraints=model.k),
            penalty_coefficient=cooper.penalty_coefficients.DensePenaltyCoefficient(
                init=torch.full((model.k,), float(init_penalty_colsum), dtype=torch.float32)
            ),
        )

        # -Q^T 1_d + 1_k - eps <= 0


    def compute_cmp_state(self):
        m = self.u_data.shape[1]
        y_pred = self.model(self.u_data)
        loss = ((self.y_data - y_pred) ** 2).sum() / m

        H = binary_entropy(self.model.Q, eps=self.model.eps)
        entropy_violation = (H - self.c0).reshape(1) # (1,)

        colsum_violation_upper = self.model.Q.sum(dim=0) - 1.0 # (20,)
        colsum_violation_lower = 1.0 - self.model.Q.sum(dim=0) # (20,)
        
        return cooper.CMPState(
            loss=loss,
            observed_constraints={
                self.entropy_eq: cooper.ConstraintState(violation=entropy_violation),
                self.colsum_eq_upper: cooper.ConstraintState(violation=colsum_violation_upper),
                self.colsum_eq_lower: cooper.ConstraintState(violation=colsum_violation_lower),
            },
        )


# =========================================================
# Debug helpers
# =========================================================

def inspect_column_gradients(cmp, col_j=0, top_rows=6):
    model = cmp.model
    eps = model.eps

    model.zero_grad(set_to_none=True)
    cmp_state = cmp.compute_cmp_state()
    cmp_state.loss.backward(retain_graph=True)
    task_col = model.Q.grad[:, col_j].detach().clone()

    with torch.no_grad():
        Q = model.Q.detach().clamp(eps, 1 - eps)
        qcol = Q[:, col_j]

        H = binary_entropy(Q, eps=eps)
        a = (H - cmp.c0).item()
        h = (Q.sum(dim=0)[col_j] - 1.0).item()

        lam_E = cmp.entropy_eq.multiplier().reshape(-1)[0].item()
        mu_j = cmp.colsum_eq.multiplier().reshape(-1)[col_j].item()

        c_E = cmp.entropy_eq.penalty_coefficient().reshape(-1)[0].item()
        c_j = cmp.colsum_eq.penalty_coefficient().reshape(-1)[col_j].item()

        entropy_deriv = torch.log((1.0 - qcol) / qcol)
        entropy_col = (lam_E + c_E * a) * entropy_deriv
        colsum_col = torch.full_like(qcol, fill_value=(mu_j + c_j * h))
        total_col = task_col + entropy_col + colsum_col

        idx = torch.argsort(total_col)[:top_rows]

        print(f"\nColumn {col_j}")
        print(
            f"a={a:+.4e}, h_j={h:+.4e}, "
            f"lambda_E={lam_E:+.4e}, mu_j={mu_j:+.4e}, "
            f"c_E={c_E:.3e}, c_j={c_j:.3e}"
        )
        print(" row | q_ij      | task       | entropy    | colsum     | total")
        for r in idx.tolist():
            print(
                f"{r:4d} | {qcol[r].item():.3e} | "
                f"{task_col[r].item():+.3e} | "
                f"{entropy_col[r].item():+.3e} | "
                f"{colsum_col[r].item():+.3e} | "
                f"{total_col[r].item():+.3e}"
            )

        print(f"#rows with total<0: {(total_col < 0).sum().item()} / {len(total_col)}")


def increase_only_colsum_penalties(
    cmp,
    col_tol=5e-2,
    growth=1.5,
    max_penalty=100.0,
):
    with torch.no_grad():
        h = (cmp.model.Q.sum(dim=0) - 1.0).abs()
        bad = h > col_tol

        if bad.any():
            p = cmp.colsum_eq.penalty_coefficient.value
            p[bad] = torch.clamp(p[bad] * growth, max=max_penalty)


# =========================================================
# Homotopy training
# =========================================================

def run_homotopy(
    arch,
    u_data,
    y_data,
    k,
    c0_start,
    beta=0.97,
    outer_iters=105,
    inner_iters=150,
    primal_lr=1e-3,
    dual_lr=1e-4,
    eps=1e-6,
    init_penalty_entropy=1.0,
    init_penalty_colsum=5.0,
    inspect_every=10,
    inspect_col=0,
):
    torch.manual_seed(42)

    d = compute_d(arch)
    print(f"Architecture: {arch}, d={d}, k={k}")
    print(f"c0_start={c0_start:.4f}, beta={beta}, outer={outer_iters}, inner={inner_iters}")

    model = SparseNNModel(arch, k, eps=eps)
    print(f"Initial entropy H(Q) = {binary_entropy(model.Q, eps).item():.4f}")

    # Build CMP and optimizers once
    cmp = SparseNNCMP(
        model=model,
        c0=c0_start,
        u_data=u_data,
        y_data=y_data,
        init_penalty_entropy=init_penalty_entropy,
        init_penalty_colsum=init_penalty_colsum,
    )

    primal_opt = torch.optim.Adam(model.parameters(), lr=primal_lr)
    dual_opt = torch.optim.SGD(cmp.dual_parameters(), lr=dual_lr, maximize=True)

    cooper_opt = cooper.optim.AlternatingPrimalDualOptimizer(
        cmp=cmp,
        primal_optimizers=primal_opt,
        dual_optimizers=dual_opt,
    )

    history = []
    c0 = c0_start

    for outer in range(outer_iters):
        cmp.c0 = c0

        for _ in range(inner_iters):
            cooper_opt.roll() # updates model parameters and CMP multipliers Q
            model.clamp_Q()

            if not torch.isfinite(model.Q).all():
                print("NaN/Inf detected in Q")
                return model, history

        # Increase only column-sum penalties every few outer iterations
        if outer > 0 and outer % 5 == 0:
            increase_only_colsum_penalties(
                cmp,
                col_tol=5e-2,
                growth=1.5,
                max_penalty=100.0,
            )

        with torch.no_grad():
            Q_now = model.Q.detach().clone()
            H_val = binary_entropy(Q_now, eps).item()
            colsum_err = (Q_now.sum(dim=0) - 1.0).abs().max().item()
            y_pred = model(u_data)
            loss_val = ((y_data - y_pred) ** 2).sum().item() / u_data.shape[1]
            gap = torch.minimum(Q_now, 1.0 - Q_now).max().item()

            cE = cmp.entropy_eq.penalty_coefficient.value[0].item()
            cCmax = cmp.colsum_eq.penalty_coefficient.value.max().item()

        history.append(
            {
                "outer": outer,
                "c0": c0,
                "H": H_val,
                "loss": loss_val,
                "gap": gap,
                "colsum_err": colsum_err,
                "cE": cE,
                "cCmax": cCmax,
            }
        )

        if outer % inspect_every == 0 or outer == outer_iters - 1:
            print(
                f"[outer={outer:03d}] c0={c0:.4f}  "
                f"H={H_val:.4f}  loss={loss_val:.6f}  "
                f"gap={gap:.4f}  colsum_err={colsum_err:.6f}  "
                f"cE={cE:.3f}  cCmax={cCmax:.3f}"
            )
            inspect_column_gradients(cmp, col_j=inspect_col, top_rows=20)

        c0 *= beta

    return model, history


# =========================================================
# Main
# =========================================================

def main():
    np.random.seed(42)

    # Data
    m = 100
    u_np = np.random.uniform(-2, 2, size=(1, m))
    y_np = u_np**3 + u_np**2 + 1.0 + 0.1 * np.random.randn(1, m)

    u_data = torch.tensor(u_np, dtype=torch.float32)
    y_data = torch.tensor(y_np, dtype=torch.float32)

    # Architecture
    arch = [1, 10, 10, 1]
    k = 20
    d = compute_d(arch)
    print(f"d = {d} (should be 120)")

    tmp_model = SparseNNModel(arch, k)
    c0_start = 0.99 * binary_entropy(tmp_model.Q).item()
    del tmp_model
    print(f"Starting c0 = {c0_start:.4f}")

    model, history = run_homotopy(
        arch=arch,
        u_data=u_data,
        y_data=y_data,
        k=k,
        c0_start=c0_start,
        beta=0.97,
        outer_iters=105,
        inner_iters=150,
        primal_lr=1e-3,
        dual_lr=1e-4,
        eps=1e-6,
        init_penalty_entropy=1.0,
        init_penalty_colsum=5.0,
        inspect_every=10,
        inspect_col=0,
    )

    Q_final = model.Q.detach().cpu().numpy()
    x_final = model.x_coeff.detach().cpu().numpy()
    w_final = Q_final @ x_final

    print("\n" + "=" * 60)
    print("FINAL RESULTS")
    print("=" * 60)

    if len(history) > 0:
        print(f"Final Q binary gap: {np.max(np.minimum(Q_final, 1.0 - Q_final)):.6f}")
        print(f"Final colsum error: {np.max(np.abs(Q_final.sum(axis=0) - 1.0)):.6f}")
        print(f"Final entropy H(Q): {binary_entropy(model.Q).item():.6f}")
        print(f"Final loss: {history[-1]['loss']:.6f}")

    print("\nEntries of Q > 0.9:")
    rows, cols = np.where(Q_final > 0.9)
    for r, c in zip(rows, cols):
        print(f"  Q[{r},{c}] = {Q_final[r, c]:.6f}")


if __name__ == "__main__":
    main()

d = 120 (should be 120)
Starting c0 = 114.5096
Architecture: [1, 10, 10, 1], d=120, k=20
c0_start=114.5096, beta=0.97, outer=105, inner=150
Initial entropy H(Q) = 115.6663


/home/ruez/venvs/jupyter-env/lib/python3.12/site-packages/torch/autograd/graph.py:869: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


[outer=000] c0=114.5096  H=65.5548  loss=12.630891  gap=0.0361  colsum_err=0.773267  cE=1.000  cCmax=5.000

Column 0
a=-4.8955e+01, h_j=-2.1503e-01, lambda_E=+0.0000e+00, mu_j=+0.0000e+00, c_E=1.000e+00, c_j=5.000e+00
 row | q_ij      | task       | entropy    | colsum     | total
   1 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   0 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   9 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   8 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   7 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   6 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   4 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   3 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   2 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
   5 | 3.673e-03 | -5.232e-02 | -2.743e+02 | -1.075e+00 | -2.754e+02
  73 | 3.871e-03 | -4.641e-0

KeyboardInterrupt: 